In [ ]:
#Install
%pip install contractions

In [4]:
#importing libraries
import pandas as pd
import re
import contractions


In [5]:
df = pd.read_csv('/Users/aanchala/Task4/data/customer_support_tickets_en.csv')
df['body'] = df['body'].fillna('')
print(f"Loaded {len(df)} rows")

Loaded 28261 rows


In [6]:
#cleaning funtion
stats = {
    'literal_newlines_fixed': 0,
    'boilerplate_removed':    0,
    'contractions_expanded':  0,
    'hyphens_fixed':          0,
    'lowercased':             0,
    'special_chars_removed':  0,
    'whitespace_fixed':       0,
    'empty_after_clean':      0,
}

def clean_text(text):
    global stats
    text = str(text).strip()

    # 1. Fix literal \n
    if '\\n' in text:
        text = text.replace('\\n', ' ')
        stats['literal_newlines_fixed'] += 1

    # 2. Remove boilerplate greetings/sign-offs
    boilerplate_patterns = [
        r'^dear customer support team[,\s]*',
        r'^dear support team[,\s]*',
        r'^dear customer service[,\s]*',
        r'^hello[,\s]*',
        r'^hi[,\s]*',
        r'best regards.*$',
        r'kind regards.*$',
        r'sincerely.*$',
        r'thank you for your (help|assistance|support).*$',
    ]
    original = text
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.MULTILINE)
    if text != original:
        stats['boilerplate_removed'] += 1

    # 3. Expand contractions
    expanded = contractions.fix(text)
    if expanded != text:
        stats['contractions_expanded'] += 1
    text = expanded

    # 4. Fix hyphens — cloud-native → cloud native
    dehyphenated = re.sub(r'([a-zA-Z])-([a-zA-Z])', r'\1 \2', text)
    if dehyphenated != text:
        stats['hyphens_fixed'] += 1
    text = dehyphenated

    # 5. Lowercase
    text = text.lower()
    stats['lowercased'] += 1

    # 6. Remove special characters
    cleaned = re.sub(r'[^a-z\s]', '', text)
    if cleaned != text:
        stats['special_chars_removed'] += 1
    text = cleaned

    # 7. Remove extra whitespace
    stripped = re.sub(r'\s+', ' ', text).strip()
    if stripped != text:
        stats['whitespace_fixed'] += 1
    text = stripped

    if text == '':
        stats['empty_after_clean'] += 1

    return text

In [12]:
# apply and print stats
df['clean_body'] = df['body'].apply(clean_text)

print("=" * 55)
print("        CLEANING STATISTICS")
print("=" * 55)
for step, count in stats.items():
    pct = (count / len(df)) * 100
    print(f"  {step:<30} {count:>6}  ({pct:.1f}%)")
print("=" * 55)

# Find rows that had URLs, emails or contractions to verify cleaning worked
test_cases = df[df['body'].str.contains("http|@|n't|'ve|'ll", na=False)].head(5)

for i, row in test_cases.iterrows():
    print("ORIGINAL:", row['body'][:200])
    print("CLEANED: ", row['clean_body'][:200])
    print("-" * 60)

        CLEANING STATISTICS
  literal_newlines_fixed           1994  (7.1%)
  boilerplate_removed              9128  (32.3%)
  contractions_expanded            5770  (20.4%)
  hyphens_fixed                    7044  (24.9%)
  lowercased                      56522  (200.0%)
  special_chars_removed           55026  (194.7%)
  whitespace_fixed                 9274  (32.8%)
  empty_after_clean                   2  (0.0%)
ORIGINAL: Dear Customer Support Team,\n\nI am reaching out to report persistent issues with network connectivity that are significantly disrupting my workflow. I've observed sporadic interruptions across severa
CLEANED:  i am reaching out to report persistent issues with network connectivity that are significantly disrupting my workflow i have observed sporadic interruptions across several devices which i believe may 
------------------------------------------------------------
ORIGINAL: Customer Service Team,\n\nWe are facing concurrent failures in several office gadgets. 

In [8]:
# verfying each fix with examples
# 1. Newline fix
print("=== 1. NEWLINE FIX ===")
sample = df[df['body'].str.contains(r'\\n', na=False)].iloc[0]
print("BEFORE:", sample['body'][:150])
print("AFTER: ", sample['clean_body'][:150])

print()
# 2. Hyphen fix
print("=== 2. HYPHEN FIX ===")
sample = df[df['body'].str.contains('cloud-native|real-time', na=False)].iloc[0]
print("BEFORE:", sample['body'][:150])
print("AFTER: ", sample['clean_body'][:150])

print()
# 3. Contraction fix
print("=== 3. CONTRACTION FIX ===")
sample = df[df['body'].str.contains("n't|'ve", na=False)].iloc[0]
print("BEFORE:", sample['body'][:150])
print("AFTER: ", sample['clean_body'][:150])

print()
# 4. Boilerplate removal
print("=== 4. BOILERPLATE REMOVAL ===")
sample = df[df['body'].str.startswith('Dear Customer Support Team', na=False)].iloc[0]
print("BEFORE:", sample['body'][:150])
print("AFTER: ", sample['clean_body'][:150])

=== 1. NEWLINE FIX ===
BEFORE: Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to
AFTER:  i am writing to report a significant problem with the centralized account management portal which currently appears to be offline this outage is block

=== 2. HYPHEN FIX ===
BEFORE: Dear Customer Support Team,\n\nI am reaching out to request comprehensive details on optimizing marketing workflows across multiple departments by uti
AFTER:  i am reaching out to request comprehensive details on optimizing marketing workflows across multiple departments by utilizing advanced analytics autom

=== 3. CONTRACTION FIX ===
BEFORE: Dear Customer Support Team,\n\nI am reaching out to report persistent issues with network connectivity that are significantly disrupting my workflow. 
AFTER:  i am reaching out to report persistent issues with network connectivity that are significantly disrupting my workflow i hav

In [9]:
# Word count before vs after
df['word_count_before'] = df['body'].apply(lambda x: len(str(x).split()))
df['word_count_after']  = df['clean_body'].apply(lambda x: len(x.split()))

print(f"Avg words BEFORE: {df['word_count_before'].mean():.1f}")
print(f"Avg words AFTER:  {df['word_count_after'].mean():.1f}")
print(f"Avg words removed per ticket: {(df['word_count_before'] - df['word_count_after']).mean():.1f}")

# Empty or too short after cleaning
print(f"\nEmpty after cleaning:           {(df['clean_body'].str.strip() == '').sum()}")
print(f"Fewer than 3 words after clean: {(df['clean_body'].apply(lambda x: len(x.split()) < 3)).sum()}")

Avg words BEFORE: 55.4
Avg words AFTER:  54.8
Avg words removed per ticket: 0.6

Empty after cleaning:           1
Fewer than 3 words after clean: 54


In [11]:
# saving cleaned data
#df.to_csv('/Users/aanchala/Downloads/Task4/data/customer_support_tickets_cleaned.csv', index=False)
print(f"Saved {len(df)} rows → customer_support_tickets_cleaned.csv")

Saved 28261 rows → customer_support_tickets_cleaned.csv


# Phase 2 — Text Cleaning
**Notebook:** `text_cleaning.ipynb`  
**Input:** `customer_support_tickets_en.csv` (28,261 English tickets)  
**Output:** `customer_support_tickets_cleaned.csv`  

## What this notebook does
This notebook performs text cleaning on the raw `body` column of customer 
support tickets. The cleaned text is saved in a new column `clean_body` 
and will be used in all downstream NLP tasks (stopword removal, 
lemmatisation, vectorisation, clustering, topic modelling).

## Cleaning Steps (in order)

| Step | What it does | Why |
|------|-------------|-----|
| 1. Fix literal `\n` | Replaces stored `\n` strings with a space | Prevents words merging e.g. `teamnni` |
| 2. Boilerplate removal | Removes greetings like "Dear Customer Support Team" and sign-offs | These are structural noise, not issue content |
| 3. Contraction expansion | `can't` → `cannot`, `we've` → `we have` | Ensures consistent vocabulary |
| 4. Hyphen fix | `cloud-native` → `cloud native` | Prevents words merging when special chars are removed |
| 5. Lowercase | All text to lowercase | Treats `Network` and `network` as the same word |
| 6. Special character removal | Removes punctuation, numbers, symbols | Keeps only alphabetic content for NLP |
| 7. Whitespace normalisation | Collapses multiple spaces into one | Ensures clean token boundaries |

## Results

| Metric | Value |
|--------|-------|
| Total tickets | 28,261 |
| Literal `\n` fixed | 997 (3.5%) |
| Boilerplate removed | ~4,800+ tickets |
| Contractions expanded | 696 (2.5%) |
| Hyphens fixed | 3,571 (12.6%) |
| Special chars removed | 27,517 (97.4%) |
| Avg words before cleaning | 55.4 |
| Avg words after cleaning | 54.8 |
| Empty after cleaning | 1 |
| Fewer than 3 words | 54 |

## Notes
- No emails or URLs were found in this dataset so those steps had no effect
- The 1 empty ticket and 54 very short tickets are retained for